In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [8]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops


Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import dagshub
import mlflow
from mlflow.tracking import MlflowClient
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = "kgord23"
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/kgord23/ML_hw_01_House-Prices.mlflow"
os.environ["MLFLOW_TRACKING_PASSWORD"] = secret_value_0

dagshub.auth.add_app_token(secret_value_0)
dagshub.init(repo_owner='kgord23', repo_name='ML_hw_01_House-Prices', mlflow=True)

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
client = MlflowClient()



The added token already exists in the token cache, skipping


Initialized MLflow to track repo "kgord23/ML_hw_01_House-Prices"

Repository kgord23/ML_hw_01_House-Prices initialized!

In [10]:
model = mlflow.sklearn.load_model("models:/HousePrices_BestModel/3")
print("done!")

done!


In [11]:
RUN_ID='ce4d9c7db2664449afab6c47491b305e'
client.download_artifacts(RUN_ID, "train_medians.csv", ".")
client.download_artifacts(RUN_ID, "train_modes.csv", ".")
client.download_artifacts(RUN_ID, "selected_cols.csv", ".")

train_medians = pd.read_csv("train_medians.csv", index_col=0).squeeze()
train_modes = pd.read_csv("train_modes.csv", index_col=0).squeeze()
selected_cols = pd.read_csv("selected_cols.csv").squeeze().tolist()

print("artifacts are downloaded!")
print(f"selected columns: {len(selected_cols)}")

artifacts are downloaded!
selected columns: 50


In [12]:
test=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
house_id=test['Id']
cols_to_drop=['Alley', 'PoolQC', 'MiscFeature', 'Fence']
test=test.drop(columns=cols_to_drop, errors='ignore')
cat_cols = [col for col in test.columns if test[col].dtype == 'object']
num_cols = [col for col in test.columns if test[col].dtype != 'object']

test[cat_cols]=test[cat_cols].fillna(train_modes)
test[num_cols]=test[num_cols].fillna(train_medians)
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

test_encoded = pd.get_dummies(test, columns=cat_cols, drop_first=True, dtype=int)

test_encoded = test_encoded.reindex(columns=selected_cols, fill_value=0)

print(f"test shape: {test_encoded.shape}")




test shape: (1459, 50)


In [13]:
Y_test_log=model.predict(test_encoded)
Y_test=np.expm1(Y_test_log)
submission = pd.DataFrame({
    'Id': house_id,
    'SalePrice': Y_test
})

submission.to_csv("submission.csv", index=False)
print("submission.csv done!")
print(submission.head())

submission.csv done!
     Id      SalePrice
0  1461  111709.049920
1  1462  148477.898134
2  1463  155736.356317
3  1464  190896.335975
4  1465  217002.002334
